In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Use environment variables already available to the kernel
import os

print("API key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

API key loaded: False


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load a local .env file if present in the notebook folder or its parent
for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path, override=False)

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is not set. Set it in your shell, VS Code environment, or add it to .env.")
if len(api_key) < 20:
    raise ValueError("ANTHROPIC_API_KEY looks too short to be valid.")

from anthropic import Anthropic

client = Anthropic(api_key=api_key)
model = "claude-sonnet-4-5"
print("Client ready")

Client ready


In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

chat(messages)


'```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n```'

In [9]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "``` json")

text =chat(messages, stop_sequences=["```"])
text


'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [10]:
import json

json.loads(text.strip())


{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [15]:
messages = []

prompt = """
Generate  three different sample GCP CLI commands. Each should be very shot.

"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()



'gcloud compute instances list\n\ngcloud storage buckets create gs://my-bucket\n\ngcloud projects describe my-project-id'

In [13]:
from IPython.display import display, Markdown

Markdown(text)



gcloud compute instances list

gcloud storage buckets create gs://my-bucket

gcloud projects describe my-project-id
